# M4.1 Biot-Savart: Loops and the Helmholtz Pair

Book pointer: chapter 11, sections 11.1 (the magnetic field) and 11.2 (magnetic forces).

## 1. Recall

Answer on paper before running any cell.

1. Write the Biot-Savart law for the field $d\mathbf{B}$ of a current element $I\,d\mathbf{l}$ at displacement $\mathbf{R}$ from the element. Which way does $d\mathbf{B}$ point relative to $d\mathbf{l}$ and $\mathbf{R}$?
2. For a circular loop, why does the field on the axis have only an axial component? Which contributions cancel, and in pairs of what?
3. Write $B_z(z)$ on the axis of a loop of radius $R$ carrying current $I$. What is it at the centre, and how does it fall off for $z \gg R$?
4. State the Helmholtz condition for two coaxial loops. Which derivatives of $B_z$ vanish at the midpoint, and why does the second one vanish?
5. A vector field is stored on a grid with 41 points along $x$, 1 along $y$ and 81 along $z$. What is the shape of the array, and what does `B[..., 2]` hold?

## 2. Principle to algorithm

The Biot-Savart law gives the field of a current element,

$$d\mathbf{B} = \frac{\mu_0 I}{4\pi}\,\frac{d\mathbf{l}\times\mathbf{R}}{|\mathbf{R}|^3}, \qquad \mathbf{R} = \mathbf{r} - \mathbf{r}',$$

and superposition adds the elements up. On a computer a wire is a polyline: an ordered list of vertices. `biot_savart.segments` turns it into midpoints $\mathbf{r}'_k$ and step vectors $d\mathbf{l}_k$, and `biot_savart.wire_B` evaluates the sum $\frac{\mu_0 I}{4\pi}\sum_k d\mathbf{l}_k\times\mathbf{R}_k/|\mathbf{R}_k|^3$ at every observation point at once. The structure is the same as the Coulomb sum of module 1: an array of observation points of shape `(N, 1, 3)` minus an array of midpoints of shape `(1, M, 3)` gives all $N\times M$ displacement vectors, `np.cross` with the broadcast `dl` gives the numerators, and a weighted `einsum` over the segment axis does the sum. Segments are processed in chunks so the $N\times M\times3$ intermediate stays bounded.

The polyline is only an approximation of the wire. The error at a distance $d$ from a segment of length $\Delta l$ scales like $(\Delta l/d)^2$, so results are trustworthy a few segment lengths away from the wire and meaningless on it. Every check below keeps that in mind.

### Animation

A Helmholtz pair seen in three dimensions. Tracer particles are released near the axis and move along $\mathbf{B}$, so their paths are field lines. Watch the region between the coils: the lines there are straight and evenly spaced, which is the uniform field the pair is built for. Outside, the lines bulge and the spacing opens up.

In [ ]:
from emlib import show
show.animation("m4_magnetostatics", "HelmholtzFieldLines")

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields, biot_savart as bs
from emlib.biot_savart import MU0

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)

### One loop in the plane through its axis

A loop of radius $R = 1$ m carrying $I = 1$ A, counter-clockwise about $+z$, is cut into 200 segments. The field is evaluated on an $x$-$z$ grid through the axis (the plane $y = 0$). The grid is built by hand because the plane is not the $x$-$y$ plane that `grids.grid2d` makes; the convention is the same, `indexing="ij"` with the first index along $x$ and the second along $z$. The arrows are drawn at unit length with colour showing $\log_{10}|\mathbf{B}|$, because the field near the wire is hundreds of times stronger than at the centre.

In [ ]:
R, I = 1.0, 1.0
mid, dl = bs.segments(bs.loop(R, 200))
print(f"{len(mid)} segments, each of length {np.linalg.norm(dl, axis=1).mean():.4f} m")

x = np.linspace(-2.5, 2.5, 81); z = np.linspace(-2.5, 2.5, 81)
X, Z = np.meshgrid(x, z, indexing="ij")
P = np.stack([X, np.zeros_like(X), Z], axis=-1)                      # (81, 81, 3): the plane y = 0
B = bs.wire_B(mid, dl, I, P)
Bmag = np.linalg.norm(B, axis=-1)
B0 = MU0 * I / (2 * R)
print(f"field at the centre {B[40, 40, 2]:.4e} T   mu0 I / 2R = {B0:.4e} T")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
st = 3
axes[0].quiver(X[::st, ::st], Z[::st, ::st], (B[..., 0] / Bmag)[::st, ::st], (B[..., 2] / Bmag)[::st, ::st],
               np.log10(Bmag / B0)[::st, ::st], cmap="plasma", pivot="mid")
axes[0].plot([R, -R], [0, 0], "o", color="C3", ms=8); axes[0].set_aspect("equal"); axes[0].set_xlabel("x"); axes[0].set_ylabel("z")
axes[0].set_title("unit arrows, colour = log10(|B| / B_centre)")
axes[1].streamplot(x, z, B[..., 0].T, B[..., 2].T, color=np.log10(Bmag / B0).T, cmap="plasma", density=1.4)
axes[1].plot([R, -R], [0, 0], "o", color="C3", ms=8); axes[1].set_aspect("equal"); axes[1].set_xlabel("x")
axes[1].set_title("field lines thread the loop and close around the wire")
plt.show()

The wire crosses the plane at $x = \pm R$ (red dots): into the page on one side, out on the other. Every field line threads the loop once and closes around one of the two wire crossings. The on-axis profile is the one case with a closed form,

$$B_z(z) = \frac{\mu_0 I R^2}{2(z^2 + R^2)^{3/2}},$$

which is $\mu_0 I/2R$ at the centre and falls as $1/z^3$ far away, the signature of a dipole.

In [ ]:
zz = np.linspace(-3, 3, 121)
B_axis = bs.wire_B(mid, dl, I, np.stack([0 * zz, 0 * zz, zz], axis=-1))
plt.plot(zz, B_axis[:, 2] / B0, "C0", lw=2, label="Biot-Savart sum")
plt.plot(zz, bs.loop_axis_B(R, I, zz) / B0, "k--", label="closed form")
tail = zz[np.abs(zz) > 1.5]
plt.plot(tail, MU0 * I * R**2 / (2 * np.abs(tail)**3) / B0, "C3:", label="1 / z^3 tail")
plt.xlabel("z / R"); plt.ylabel("B_z / (mu0 I / 2R)"); plt.legend(); plt.title("on the axis of one loop"); plt.show()
print(f"transverse components on the axis, max |B_x|, |B_y|: {np.abs(B_axis[:, :2]).max():.1e} T  (symmetry says zero)")

### The Helmholtz pair

Two coaxial loops carrying the same current in the same sense, separated by $s$. By symmetry $B_z$ is even about the midpoint, so its first derivative vanishes there for any $s$. The second derivative is $\propto (s^2 - R^2)$ and vanishes only for $s = R$: that is the Helmholtz condition, and it makes the field flat to fourth order in $z$. The cell compares $s = R/2$ (a dip-free but curved maximum), $s = R$ (flat), and $s = 2R$ (a dip in the middle because the loops are too far apart to help each other).

In [ ]:
def pair(R, s, n=200):
    m1, d1 = bs.segments(bs.loop(R, n, (0, 0, -s / 2)))
    m2, d2 = bs.segments(bs.loop(R, n, (0, 0, +s / 2)))
    return np.vstack([m1, m2]), np.vstack([d1, d2])

zz = np.linspace(-1, 1, 201)
axis_pts = np.stack([0 * zz, 0 * zz, zz], axis=-1)
for s, name in ((0.5 * R, "s = R/2"), (R, "s = R (Helmholtz)"), (2 * R, "s = 2R")):
    m, d = pair(R, s)
    Bz = bs.wire_B(m, d, I, axis_pts)[:, 2]
    plt.plot(zz, Bz / Bz[100], label=name)
plt.axvspan(-0.25, 0.25, color="grey", alpha=0.15, label="|z| < R/4")
plt.xlabel("z / R"); plt.ylabel("B_z / B_z(0)"); plt.legend(); plt.title("two loops: the field between them"); plt.show()

m_h, d_h = bs.helmholtz(R, 200)
Bz_h = bs.wire_B(m_h, d_h, I, axis_pts)[:, 2]
B_centre = (4 / 5) ** 1.5 * MU0 * I / R
print(f"Helmholtz centre: {Bz_h[100]:.5e} T   (4/5)^(3/2) mu0 I / R = {B_centre:.5e} T")
print(f"deviation of B_z from the centre value at |z| = R/4: {abs(Bz_h[125] / Bz_h[100] - 1):.1e},  at |z| = R/2: {abs(Bz_h[150] / Bz_h[100] - 1):.1e}")

The flatness is not confined to the axis. The map below shows $|\mathbf{B}|/B(0) - 1$ in percent over the plane $y = 0$ inside the pair. The region where the field is uniform to within one percent is roughly a sphere of radius $R/3$ around the centre, which is the working volume of a laboratory Helmholtz coil.

In [ ]:
xg = np.linspace(-0.9, 0.9, 73); zg = np.linspace(-0.9, 0.9, 73)
Xg, Zg = np.meshgrid(xg, zg, indexing="ij")
Pg = np.stack([Xg, np.zeros_like(Xg), Zg], axis=-1)
Bg = bs.wire_B(m_h, d_h, I, Pg)
dev = 100 * (np.linalg.norm(Bg, axis=-1) / Bz_h[100] - 1)
fig, ax = plt.subplots(figsize=(6.5, 6))
cs = ax.contourf(Xg, Zg, dev, levels=[-20, -10, -5, -2, -1, -0.5, -0.1, 0.1, 0.5, 1, 2, 5, 10, 20], cmap="coolwarm", extend="both")
plt.colorbar(cs, label="(|B| / B(0) - 1)  [%]")
ax.contour(Xg, Zg, np.abs(dev), levels=[1.0], colors="k")
ax.axhline(0.5, color="C3", ls=":"); ax.axhline(-0.5, color="C3", ls=":")
ax.set_aspect("equal"); ax.set_xlabel("x / R"); ax.set_ylabel("z / R"); ax.set_title("Helmholtz pair: the black line encloses the 1 % region\n(red dotted: planes of the two coils)")
plt.show()
r_grid = np.hypot(Xg, Zg)
print(f"largest |deviation| inside r < R/3: {np.abs(dev[r_grid < R / 3]).max():.2f} %   inside r < R/2: {np.abs(dev[r_grid < R / 2]).max():.2f} %")

### Exercise: a square loop

Replace the body of `square_loop_segments` so that it returns the midpoints and step vectors of a square loop of side $a$ lying in the $x$-$y$ plane, centred on the origin and traversed counter-clockwise seen from $+z$, with $n$ segments per side. The check compares the field at the centre with the closed form for a square, $B_z = \dfrac{2\sqrt2\,\mu_0 I}{\pi a}$, which comes from adding four finite-wire results (`biot_savart.finite_wire_B` with $r = a/2$, $L = a/2$).

In [ ]:
def square_loop_segments(a, n=200):
    """(mid, dl) of a square loop of side a in the x-y plane, counter-clockwise about +z, n segments per side."""
    corners = 0.5 * a * np.array([[-1, -1, 0], [1, -1, 0], [1, 1, 0], [-1, 1, 0]], dtype=float)   # replace from here with your own version
    verts = np.vstack([bs.straight_wire(corners[k], corners[(k + 1) % 4], n)[:-1] for k in range(4)])
    return bs.segments(verts, closed=True)


a = 1.0
m_sq, d_sq = square_loop_segments(a)
B_sq = bs.wire_B(m_sq, d_sq, I, np.array([0.0, 0.0, 0.0]))
B_sq_exact = 2 * np.sqrt(2) * MU0 * I / (np.pi * a)
print(f"square loop, centre: B_z = {B_sq[2]:.5e} T, B_x = {B_sq[0]:.1e}, B_y = {B_sq[1]:.1e}   closed form B_z = {B_sq_exact:.5e} T   (= 4 x finite wire: {4 * bs.finite_wire_B(I, a / 2, a / 2):.5e})")
print(f"total length of the polyline: {np.linalg.norm(d_sq, axis=1).sum():.4f}  (should be 4a = {4 * a})")
assert np.isclose(B_sq[2], B_sq_exact, rtol=1e-4) and np.allclose(B_sq[:2], 0.0, atol=1e-15)

## 4. Verify

Three closed forms.

On the axis: the Biot-Savart sum for a 400-segment loop against $\mu_0 I R^2 / 2(z^2 + R^2)^{3/2}$ at several heights. The polyline replaces each arc by a chord whose midpoint lies just inside the circle, so at the centre the sum is slightly high; with 400 segments the chord is $0.016R$ and the error is below $10^{-4}$.

Far field: at distances of $20R$ and beyond, the loop is a magnetic dipole with moment $m = I\pi R^2$ along $+z$, and `biot_savart.dipole_B` gives $\frac{\mu_0}{4\pi}\frac{3(\mathbf{m}\cdot\hat{\mathbf{r}})\hat{\mathbf{r}} - \mathbf{m}}{r^3}$. The next term in the expansion is smaller by $(R/r)^2$, so at $20R$ the agreement should be a few parts in a thousand.

Helmholtz centre: $B = (4/5)^{3/2}\mu_0 I/R$, which is two loops each contributing $\mu_0 I R^2/2(R^2/4 + R^2)^{3/2}$.

In [ ]:
mid4, dl4 = bs.segments(bs.loop(R, 400))
z_test = np.array([0.0, 0.25, 0.5, 1.0, 2.0, 4.0])
B_num = bs.wire_B(mid4, dl4, I, np.stack([0 * z_test, 0 * z_test, z_test], axis=-1))[:, 2]
B_ex = bs.loop_axis_B(R, I, z_test)
rel_axis = np.abs(B_num / B_ex - 1)
print("on axis, z / R:", z_test); print("signed relative error:", np.array2string(B_num / B_ex - 1, formatter={"float": lambda v: f"{v:+.1e}"}))
assert rel_axis.max() < 1e-4

m_vec = np.array([0.0, 0.0, I * np.pi * R**2])
far = np.array([[20.0, 0.0, 0.0], [0.0, 0.0, 20.0], [15.0, 10.0, 12.0], [-20.0, 5.0, -18.0], [0.0, 25.0, 25.0]])
B_far = bs.wire_B(mid4, dl4, I, far); B_dip = bs.dipole_B(m_vec, far)
rel_far = np.linalg.norm(B_far - B_dip, axis=1) / np.linalg.norm(B_dip, axis=1)
print("far field, |r| / R:", np.linalg.norm(far, axis=1)); print("relative difference from a point dipole:", rel_far)
assert rel_far.max() < 1e-2

Bz_c = bs.wire_B(*bs.helmholtz(R, 400), I, np.array([0.0, 0.0, 0.0]))[2]
print(f"Helmholtz centre: {Bz_c:.6e} T   closed form {B_centre:.6e} T   relative error {abs(Bz_c / B_centre - 1):.1e}")
assert abs(Bz_c / B_centre - 1) < 1e-4

plt.loglog(z_test[1:], B_ex[1:] / B0, "k-", label="closed form"); plt.loglog(z_test[1:], B_num[1:] / B0, "C0o", label="Biot-Savart sum")
plt.loglog(z_test[1:], (R / z_test[1:])**3, "C3:", label="(R/z)^3")
plt.xlabel("z / R"); plt.ylabel("B_z / B_centre"); plt.legend(); plt.title("the loop becomes a dipole"); plt.show()

## 5. Explore and rebuild

Explore. Reverse the current in one coil of the Helmholtz pair. The field at the centre is now zero and $B_z$ grows linearly with $z$: this is the anti-Helmholtz or gradient coil used in magnetic traps. Compute $B_z(z)$ along the axis for separations $s = R/2$, $R$, $\sqrt3 R$ and $2R$, fit the slope at the centre, and measure the range of $z$ over which the field stays within one percent of the linear fit. The separation $\sqrt3 R$ should stand out: find out which derivative it kills, by the same reasoning that gave the Helmholtz condition. Then look at the field off the axis and check that $\nabla\cdot\mathbf{B} = 0$ forces the transverse gradient to be minus half the axial one.

Rebuild. Close this notebook. On a blank editor, write `wire_B` from scratch (segments to midpoints and steps, the broadcast cross product, the chunked sum) and make it pass `tests/test_biot_savart.py`. Then reproduce the three-separation comparison of the pair and the $(4/5)^{3/2}$ check.